# Week 64, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w064d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ast
tree = ast.parse("total = price * (1 + rate)")
print(sorted(n.id for n in ast.walk(tree) if isinstance(n, ast.Name)))
print(type(tree.body[0].value.right).__name__)

**Predict before you run:** what will this print?



In [ ]:
stack = []
for op in [("PUSH", 6), ("PUSH", 2), ("SUB",), ("PUSH", 3), ("MUL",)]:
    if op[0] == "PUSH":
        stack.append(op[1])
    else:
        b, a = stack.pop(), stack.pop()
        stack.append(a - b if op[0] == "SUB" else a * b)
print(stack)

**Predict before you run:** what will this print?



In [ ]:
import sys
def a():
    return b()
def b():
    f = sys._getframe()
    return [f.f_code.co_name, f.f_back.f_code.co_name]
print(a())

**Predict before you run:** what will this print?



In [ ]:
import sys
s1 = "abc" * 10
s2 = "ab€" * 10
print(sys.getsizeof(s1 + "a") - sys.getsizeof(s1), sys.getsizeof(s2 + "a") - sys.getsizeof(s2))

**Predict before you run:** what will this print?



In [ ]:
import sys, types
m = types.ModuleType("fake")
m.value = 42
sys.modules["fake"] = m
from fake import value
print(value)
del sys.modules["fake"]

### Exercise: Inside a closure

Write `closure_vars(func)`: a dict from each **free variable** of a function (the names in `func.__code__.co_freevars`) to its current value, read from the matching cell in `func.__closure__` (`cell.cell_contents`; use `None` for an empty cell, which raises `ValueError`). Closures and `nonlocal` were Week 60; this is what they look like inside.

In [ ]:
def closure_vars(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_closure_vars():\n    \"closures and code objects\"\n    def make(rate):\n        count = 0\n        def bump(x):\n            nonlocal count\n            count += 1\n            return x * rate\n        return bump\n    f = make(3)\n    assert closure_vars(f) == {\"count\": 0, \"rate\": 3}\n    f(2)\n    f(2)\n    assert closure_vars(f)[\"count\"] == 2 and closure_vars(f)[\"rate\"] == 3\n    assert closure_vars(lambda: 1) == {}")

### Exercise: Walking a tree with a generator

Write a generator `walk_tree(node, depth=0)` (Week 59) that yields `(depth, node type name)` for every node of an AST in **pre-order** (a node before its children, children in the order `ast.iter_child_nodes` gives), using `yield from` for the recursion. Then `deepest(source)`: the greatest depth in the tree of the parsed source.

In [ ]:
import ast

def walk_tree(node, depth=0):
    ___

def deepest(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_walk_tree():\n    \"a generator over a syntax tree\"\n    import ast, types\n    tree = ast.parse(\"x = 1 + y\")\n    gen = walk_tree(tree)\n    assert isinstance(gen, types.GeneratorType)\n    assert list(walk_tree(tree)) == [(0, \"Module\"), (1, \"Assign\"), (2, \"Name\"), (3, \"Store\"), (2, \"BinOp\"),\n                                     (3, \"Constant\"), (3, \"Add\"), (3, \"Name\"), (4, \"Load\")]\n    assert deepest(\"a\") == 3 and deepest(\"f(g(h(1)))\") > deepest(\"f(1)\")")

### Exercise: Forge: a code checker

Write `check` and `report` to the specification above.

In [ ]:
import ast
import builtins
import io
import tokenize

def check(source):
    ___

def report(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_imports_and_locals():\n    \"unused imports and locals\"\n    src = (\n        \"import os\\n\"\n        \"import sys, json\\n\"\n        \"import collections.abc\\n\"\n        \"from math import pi, tau as full_turn\\n\"\n        \"__all__ = ['full_turn']\\n\"\n        \"def area(r):\\n\"\n        \"    unused = 2\\n\"\n        \"    total = pi * r * r\\n\"\n        \"    _ = 0\\n\"\n        \"    return total\\n\"\n        \"def counter():\\n\"\n        \"    count = 0\\n\"\n        \"    def step():\\n\"\n        \"        return count + 1\\n\"\n        \"    return step\\n\"\n        \"print(sys.argv)\\n\"\n    )\n    assert check(src) == [\n        (1, \"F401\", \"'os' imported but unused\"),\n        (2, \"F401\", \"'json' imported but unused\"),\n        (3, \"F401\", \"'collections' imported but unused\"),\n        (7, \"F841\", \"local variable 'unused' is assigned to but never used\"),\n    ]\n    assert check(\"import a.b\\nprint(a.b.c)\\n\") == []\n    assert check(\"def f():\\n    global g\\n    g = 1\\n\") == []\n\ndef test_other_checks():\n    \"defaults, excepts, built-ins and None\"\n    src = (\n        \"def add(item, items=[], seen=set(), *, opts={}):\\n\"\n        \"    if item == None:\\n\"\n        \"        return items\\n\"\n        \"    try:\\n\"\n        \"        items.append(item)\\n\"\n        \"    except:\\n\"\n        \"        pass\\n\"\n        \"    return items\\n\"\n        \"list = [1]\\n\"\n        \"def max(id):\\n\"\n        \"    return id != None\\n\"\n    )\n    assert check(src) == [\n        (1, \"B006\", \"mutable default argument\"),\n        (2, \"E711\", \"comparison to None should use 'is' or 'is not'\"),\n        (6, \"E722\", \"do not use bare 'except'\"),\n        (9, \"A001\", \"'list' shadows a built-in\"),\n        (10, \"A001\", \"'id' shadows a built-in\"),\n        (10, \"A001\", \"'max' shadows a built-in\"),\n        (11, \"E711\", \"comparison to None should use 'is' or 'is not'\"),\n    ]\n    assert check(\"def f(x=None, y=(), z=0):\\n    return x is None\\n\") == []\n\ndef test_noqa_and_report():\n    \"suppression, syntax errors and reports\"\n    src = \"import os  # noqa\\nimport re  # noqa: E711\\nx = 1 == None  # noqa: E711, F401\\ndef f(a=[]):  # NOQA: B006\\n    pass\\n\"\n    assert check(src) == [(2, \"F401\", \"'re' imported but unused\")]\n    assert check(\"x = = 1\") == [(1, \"E999\", \"syntax error\")]\n    assert report(\"import os\\nlist = 3\\n\") == \"1: F401 'os' imported but unused\\n2: A001 'list' shadows a built-in\"\n    assert report(\"x = 1\\n\") == \"\"\n    text = \"s = '# noqa'\\nimport os\\n\"\n    assert check(text) == [(2, \"F401\", \"'os' imported but unused\")]")

> **Programs that read programs.** The checker never runs the code it checks, yet it finds real bugs, because the syntax tree holds the program's structure precisely. That's the idea behind every tool in this family: linters (flake8, pylint, Ruff), formatters (Black), type checkers (mypy, Week 33), refactoring tools and IDE features such as "find all references". Each starts from the same stages you met on Monday, tokens for what the tree forgets (comments, exact layout) and the AST for everything else, and adds a symbol table when it needs to know which name means what. When a tool reports a strange warning, you can now picture what it saw.

### Exercise: Forge bonus: line coverage

**Coverage** tools show which lines of a program ran, so you can find code your tests never reach. Write:

- `statement_lines(source)`: the sorted line numbers where statements start (every `ast.stmt` node's `lineno`), leaving out docstrings and other string-only expression statements;
- `measure(source, namespace=None)`: compile the source with the filename `"<covered>"`, run it in `namespace` (a new dict if none) with a tracer that counts `"line"` events in frames from that filename only (module level and functions), and return a dict with `"statements"`, `"executed"` (statement lines that ran, sorted), `"missed"` (those that didn't), `"percent"` (executed statements as a rounded whole-number percentage, 100 when there are none) and `"hits"` (for each executed statement line, how many times it ran). Restore the previous tracer even if the code raises;
- `annotate(source, result)`: the source with a three-character margin and `" | "` before each line: the hit count (right-aligned) for executed statement lines, `!` for missed ones, and spaces for other lines.

In [ ]:
import ast
import sys

def statement_lines(source):
    ___

def measure(source, namespace=None):
    ___

def annotate(source, result):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_measure():\n    \"line coverage\"\n    src = (\n        \"def grade(score):\\n\"\n        \"    if score >= 50:\\n\"\n        \"        return 'pass'\\n\"\n        \"    return 'fail'\\n\"\n        \"results = []\\n\"\n        \"for s in (70, 80):\\n\"\n        \"    results.append(grade(s))\\n\"\n    )\n    ns = {}\n    r = measure(src, ns)\n    assert ns[\"results\"] == [\"pass\", \"pass\"]\n    assert r[\"statements\"] == [1, 2, 3, 4, 5, 6, 7] and r[\"missed\"] == [4]\n    assert r[\"executed\"] == [1, 2, 3, 5, 6, 7] and r[\"percent\"] == 86\n    assert r[\"hits\"][2] == 2 and r[\"hits\"][7] == 2 and r[\"hits\"][1] == 1\n    full = measure(\"x = 1\\ny = x + 1\\n\")\n    assert full[\"percent\"] == 100 and full[\"missed\"] == []\n    assert statement_lines('\"\"\"doc\"\"\"\\nx = 1\\n') == [2]\n\ndef test_annotate():\n    \"annotated listing\"\n    src = \"a = 1\\nif a > 5:\\n    a = 0\\nb = a\\n\"\n    r = measure(src)\n    assert annotate(src, r).split(\"\\n\") == [\"  1 | a = 1\", \"  1 | if a > 5:\", \"  ! |     a = 0\", \"  1 | b = a\", \"    | \"]\n    import sys\n    before = sys.gettrace()\n    try:\n        measure(\"raise ValueError('x')\\n\")\n        assert False\n    except ValueError:\n        pass\n    assert sys.gettrace() is before")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [how CPython runs your code](w064d1.html), [bytecode and dis](w064d2.html), [frames and the evaluation loop](w064d3.html), [objects in C](w064d4.html) or [the import system](w064d5.html).

## Mixed review

## Forge: a code checker

A **linter** reads code without running it and reports likely mistakes. Write `check(source)`, returning a sorted list of `(line, code, message)` tuples with no duplicates, and `report(source)`, the same problems as lines `"line: CODE message"` joined by newlines (an empty string if there are none). Checks:

| code | finds | message |
|---|---|---|
| `F401` | an imported name that's never read anywhere (`import a.b` binds `a`; `as` names count; names listed in a module-level `__all__` count as used) | `'os' imported but unused` |
| `F841` | inside a function, a name assigned with `=` anywhere in the function's body that's never read in that body, ignoring `_` and names declared `global` or `nonlocal`; report the line of the first assignment | `local variable 'x' is assigned to but never used` |
| `B006` | a default argument that's a list, dict or set display, or a call to `list`, `dict` or `set` (positional and keyword-only defaults) | `mutable default argument` |
| `E722` | `except:` with no exception type | `do not use bare 'except'` |
| `A001` | a name that's assigned, or a function, class or parameter name, that's also a built-in (`dir(builtins)`, ignoring names starting with `_`) | `'list' shadows a built-in` |
| `E711` | `== None` or `!= None` | `comparison to None should use 'is' or 'is not'` |

A line with a `# noqa` comment hides every problem on that line; `# noqa: E711, F401` hides only those codes (case-insensitive `noqa`). Find comments with `tokenize`, so `"# noqa"` inside a string doesn't count. Code with a syntax error gives `[(line, "E999", "syntax error")]`.

## Forge bonus: line coverage

## Weekly quiz

## Week 64 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can follow code through tokens, the AST and code objects, and analyse and rewrite code with ast. ([Day 1](w064d1.html))
- I can read dis output and build and compile for a stack machine with jumps. ([Day 2](w064d2.html))
- I can inspect frames and tracebacks and trace calls and lines with sys.settrace safely. ([Day 3](w064d3.html))
- I can explain the object header, type slots, big integers, compact strings and list growth. ([Day 4](w064d4.html))
- I can explain each step of an import and write my own finder and loader, including packages. ([Day 5](w064d5.html))
- I built a code checker and a line coverage tool. (Today)

Tomorrow is a **rest day**. Next week: **Performance**: profiling, algorithms versus constant factors, fast data structures, caching, and vectorisation with NumPy.